# Conductive-substrate GPU benchmark (Colab)

Select **Runtime > Change runtime type > T4 GPU**. Set `BENCH_SHA` to the full commit to
measure (the notebook asserts HEAD equals it). Nothing is pip-installed: Colab's own CuPy is
used and recorded. Results are machine-readable JSON (`results-<sha7>.json`), plus an
optional Nsight Systems kernel census.

Record the GPU model printed below with every result. **If Colab assigns anything other than
a Tesla T4, do not compare its timings with the T4 baseline.** See
`tools/colab/substrate_benchmark.py` for what each section measures, and
`docs/validation/CONSOLIDATION-AUDIT.md` for the recorded results.

In [ ]:
import os, subprocess
BENCH_SHA = ""          # full 40-character commit, e.g. the one quoted in the audit
BENCH_ARGS = "all"      # or e.g. "end_to_end counts thomas"
assert len(BENCH_SHA) == 40, "set BENCH_SHA to a full commit hash"
print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
                               "--format=csv,noheader"], text=True))

In [ ]:
# Clone, pin, and launch detached (survives a disconnected cell).
import urllib.request
os.environ["BENCH_SHA"], os.environ["BENCH_ARGS"] = BENCH_SHA, BENCH_ARGS
src = urllib.request.urlopen("https://raw.githubusercontent.com/AlexandreEros/Palintropos/"
                             f"{BENCH_SHA}/tools/colab/substrate_benchmark_colab.py").read().decode()
exec(compile(src, "substrate_benchmark_colab.py", "exec"))

In [ ]:
# Progress (re-run until it prints DONE; the full sweep takes about an hour on a T4).
os.environ["BENCH_POLL"] = "1"
try:
    exec(compile(src, "substrate_benchmark_colab.py", "exec"))
except SystemExit:
    pass
finally:
    del os.environ["BENCH_POLL"]

In [ ]:
# Optional: kernel-launch census under Nsight Systems (after the benchmark has finished).
r = subprocess.run(["python3", "tools/colab/substrate_nsys.py", "--out",
                    f"/content/bench/nsys-{BENCH_SHA[:7]}.json", "--steps", "20",
                    "--state-file", "/content/bench/state.npz"],
                   cwd="/content/bench/Palintropos", env={**os.environ, "PYTHONPATH": "src:."},
                   capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-2000:])

In [ ]:
from google.colab import files
for name in (f"results-{BENCH_SHA[:7]}.json", f"nsys-{BENCH_SHA[:7]}.json"):
    p = f"/content/bench/{name}"
    if os.path.exists(p):
        files.download(p)